# Probabilistic Analysis and Randomized Algorithms

Probabilistic analysis studies expected behavior when some part of the model is random. The randomness may come from the input order, from a probability distribution over inputs, or from random choices made by the algorithm itself.

A randomized algorithm deliberately uses random choices during execution. This notebook keeps those two ideas separate: first we analyze deterministic hiring rules under random arrival orders, then we use simulation to check the analysis and build intuition for randomized algorithms.

Main text source: Cormen, Leiserson, Rivest, and Stein, *Introduction to Algorithms*, 4th edition, Chapter 5, [MIT Press](https://mitpress.mit.edu/9780262367509/introduction-to-algorithms/).


## Learning Goals

By the end of this notebook you should be able to:

- distinguish probabilistic analysis from randomized algorithm design;
- classify Las Vegas and Monte Carlo randomized algorithms;
- use indicator random variables and linearity of expectation in a simple algorithm analysis;
- derive why the hiring-assistant algorithm hires about `H_n = Theta(log n)` times on a random input order;
- explain the classical secretary problem and the `n/e` look-then-leap rule;
- compare theoretical probabilities with Monte Carlo simulation results.


## Types of Randomized Algorithms

### Las Vegas Algorithms

A Las Vegas algorithm always returns a correct answer, but its running time may depend on random choices. Randomized quicksort is the standard example: the sorted output is correct, while random pivots make the expected running time good on every fixed input.

### Monte Carlo Algorithms

A Monte Carlo algorithm has a bounded running time, but it may return an incorrect answer with small probability. The error can be one-sided, where only one kind of answer can be wrong, or two-sided, where either answer might be wrong.

Examples include Freivalds' algorithm for checking matrix multiplication and probabilistic primality tests such as Miller-Rabin. Independent repetition is a common way to reduce the error probability.

### Probabilistic Analysis Without a Randomized Algorithm

The hiring-assistant algorithm below is deterministic once the candidate order is fixed. The analysis is probabilistic because we assume the candidates arrive in a uniformly random order.


In [ ]:
from __future__ import annotations

from collections import Counter
import math
import random
import statistics

try:
    import numpy as np
except Exception:
    np = None

try:
    import pandas as pd
except Exception:
    pd = None

try:
    import matplotlib.pyplot as plt
except Exception:
    plt = None

try:
    from IPython import get_ipython
    from IPython.display import display
    import ipywidgets as widgets
    NOTEBOOK_UI = get_ipython() is not None
except Exception:
    display = print
    widgets = None
    NOTEBOOK_UI = False

WIDGETS_AVAILABLE = widgets is not None and NOTEBOOK_UI
SEED = 2025
random.seed(SEED)
rng = random.Random(SEED)


def harmonic_number(n: int) -> float:
    """Return H_n = 1 + 1/2 + ... + 1/n."""
    if n < 1:
        return 0.0
    return sum(1 / k for k in range(1, n + 1))


def show_table(rows):
    """Display rows nicely when pandas is available, otherwise return plain data."""
    if pd is not None:
        return pd.DataFrame(rows)
    return rows


print(f"Using random seed {SEED}")


## Probabilistic Analysis Toolkit

For the first analysis, we only need two ideas.

An **indicator random variable** is `1` when an event happens and `0` otherwise. If `I_i` is `1` when candidate `i` is hired, then the total number of hires is:

`X = I_1 + I_2 + ... + I_n`.

By **linearity of expectation**:

`E[X] = E[I_1] + E[I_2] + ... + E[I_n]`.

In a random permutation, candidate `i` is hired exactly when they are better than all candidates before them. Among the first `i` candidates, each is equally likely to be the best, so:

`P(candidate i is hired) = 1/i`.

Therefore the expected number of hires is:

`E[X] = 1 + 1/2 + 1/3 + ... + 1/n = H_n = Theta(log n)`.


In [ ]:
def count_records(order):
    """Count how many times a new maximum appears while scanning left to right."""
    best_so_far = -math.inf
    records = 0
    for value in order:
        if value > best_so_far:
            records += 1
            best_so_far = value
    return records


def simulate_record_counts(n: int, trials: int = 10_000, seed: int = SEED):
    local_rng = random.Random(seed)
    base_order = list(range(n))
    counts = []
    for _ in range(trials):
        order = base_order[:]
        local_rng.shuffle(order)
        counts.append(count_records(order))
    return {
        "n": n,
        "trials": trials,
        "empirical_mean_hires": sum(counts) / trials,
        "theoretical_H_n": harmonic_number(n),
        "min_hires_seen": min(counts),
        "max_hires_seen": max(counts),
    }


record_rows = [simulate_record_counts(n, trials=5_000) for n in (10, 25, 50, 100, 250)]
show_table(record_rows)


In [ ]:
if plt is not None:
    xs = [row["n"] for row in record_rows]
    empirical = [row["empirical_mean_hires"] for row in record_rows]
    theory = [row["theoretical_H_n"] for row in record_rows]

    plt.figure(figsize=(7, 4))
    plt.plot(xs, empirical, "o-", label="simulation")
    plt.plot(xs, theory, "s--", label="H_n theory")
    plt.xlabel("number of candidates")
    plt.ylabel("expected number of hires")
    plt.title("Hiring assistant: records in a random order")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()
else:
    print("matplotlib is not available, skipping plot")


## Hire Assistant problem

Suppose you need to hire a new office assistant but your previous attempts have been unsuccessful. To solve this problem, you decide to use an employment agency that will send you one candidate each day for an interview. You have to pay a small fee to the agency for each interview, and hiring an applicant is even more costly as it requires firing your current office assistant and paying a substantial hiring fee to the agency. Since you are committed to having the best possible person for the job, you have decided that if a candidate is more qualified than your current assistant, you will hire the new candidate and fire the current assistant. Although you are willing to pay the resulting cost, you want to estimate the price of this strategy.


Source notes for this section:

- CLRS Chapter 5.1 presents the hiring-assistant problem as the standard introduction to probabilistic analysis.
- The broader hiring variants later in the course can use Sergei Vassilvitskii's Stanford slide deck, [The Hiring Problem: Going Beyond Secretaries](https://theory.stanford.edu/~sergei/slides/hiring-dagstuhl.pdf), summarized in `sources/randomized_algorithms/hiring_problem.md`.


## Trivial Case - Instant Interviews and No Replacement Costs

If interviews and replacements are free, we can scan all candidates and return the maximum score in `O(n)` time. The interesting question appears when each interview and each replacement has a cost, because then the number of times we hire matters.


### Hire Assistant Problem Pseudocode

The basic online replacement rule is:

1. Set the current best candidate to `None` and the current best score to negative infinity.
2. Interview candidates one at a time.
3. If the new candidate is better than the current best candidate:
   - hire the new candidate;
   - if there was already an assistant, pay the replacement/firing cost;
   - update the current best candidate and score.
4. Otherwise, reject the candidate.
5. Return the total cost and the best candidate hired.

The exact scoring system is part of the model. In the code below, a candidate is represented directly by a numeric score.


In [ ]:
def score(candidate):
    """In this teaching model, the candidate object is already its score."""
    return candidate


def is_hirable(candidate_score, index, num_candidates, best_score, hire_cost, fire_cost, total_cost, max_budget):
    """Hire whenever the current candidate is the best seen so far."""
    return candidate_score > best_score


def hire_assistant(
    candidates,
    is_hirable_fun=is_hirable,
    hire_cost=0,
    fire_cost=0,
    interview_cost=0,
    max_budget=0,
    max_hires=None,
    debug=False,
):
    """Run an online hiring rule and return total cost, best candidate, and best score."""
    best_candidate = None
    best_score = -math.inf
    total_cost = 0
    hire_count = 0
    n = len(candidates)

    for index, candidate in enumerate(candidates):
        candidate_score = score(candidate)
        total_cost += interview_cost

        if is_hirable_fun(candidate_score, index, n, best_score, hire_cost, fire_cost, total_cost, max_budget):
            if best_candidate is not None:
                total_cost += fire_cost
                if debug:
                    print(f"Paying {fire_cost} to fire candidate with score {best_score}")

            total_cost += hire_cost
            best_candidate = candidate
            best_score = candidate_score
            hire_count += 1

            if debug:
                print(f"Hiring candidate with score {candidate_score}, paying {hire_cost}")
                print(f"Total cost now is {total_cost}")

            if max_hires is not None and hire_count >= max_hires:
                break

    return total_cost, best_candidate, best_score


In [ ]:
random.seed(SEED)
candidates = [round(random.random(), 4) for _ in range(20)]
candidates


In [ ]:
# let's run the trivial default case - no costs and we have O(n) complexity
hire_assistant(candidates)


## Knowing the random distribution

In the above example since we know the maximum (1.0) we could premake a rule - heuristic say any candidate over 0.99 is amazing and stop early.

However, this range of scores most likely will not be available in a real life situation.


In [ ]:
# how many candidates do we have?
print(f"We have {len(candidates)} candidates")


In [ ]:
hire_assistant(candidates,
               is_hirable_fun=is_hirable,
               hire_cost=2_000,
               fire_cost=5_000,
               interview_cost=250)


## Sorted candidates


In [ ]:
# employment agency gives you a list of candidates in already sorted order,
# sadly for you it is ascending and you do not realize that
sorted_candidates = sorted(candidates)
hire_assistant(sorted_candidates, is_hirable_fun=is_hirable,
               hire_cost=2_000,
               fire_cost=5_000,
               interview_cost=200)


In [ ]:
# let's make a new is_hirable function that will simply check for improvement and also for threshold score say 0.9
def is_hirable_thresh(candidate, index, num_candidates, best_score, hire_cost, fire_cost, total_cost, max_budget, threshold=0.9):
    return candidate > best_score and candidate > threshold

hire_assistant(sorted_candidates, is_hirable_fun=is_hirable_thresh,
                hire_cost=2_000,
                fire_cost=5_000,
                interview_cost=200)


In [ ]:
# now let's hire assistant when max_hires is 1
hire_assistant(sorted_candidates, is_hirable_fun=is_hirable_thresh,
                hire_cost=2_000,
                fire_cost=5_000,
                interview_cost=200,
                max_hires=1)
# if we are allowed to only hire one person then we stop early


In [ ]:
def make_hirable_function_with_threshold(threshold_value):
    """Function factory to create an is_hirable function with a specific threshold."""
    def is_hirable_with_custom_thresh(candidate, index, num_candidates, best_score, hire_cost, fire_cost, total_cost, max_budget):
        return candidate > best_score and candidate > threshold_value
    return is_hirable_with_custom_thresh

# Example usage:
thresh_07_hirable_func = make_hirable_function_with_threshold(0.7)
hire_assistant(candidates, is_hirable_fun=thresh_07_hirable_func, hire_cost=2_000, fire_cost=5_000, interview_cost=200)


### Threshold Rules

If the score distribution is known, a fixed threshold can be a reasonable decision rule: hire the first candidate whose score is high enough. The rule itself is deterministic, but its cost and success probability still need probabilistic analysis because candidate order and candidate scores may be random.


### Hire Assistant Implementation Notes

The implementation scans candidates once. Every candidate costs one interview. A candidate is hired when the selected rule returns `True`. The first hire pays only the hiring cost; later hires also pay the firing cost for replacing the current assistant.

With the default rule, every new record is hired. On a random candidate order, the expected number of such records is `H_n`, so the expensive part of the process grows like `Theta(log n)` in expectation, not `Theta(n)`.


## Monte Carlo Simulation

Monte Carlo Method is a computational algorithm that uses random sampling to estimate the solutions to problems in various fields such as physics, engineering, finance, and computer science. It is named after the famous Monte Carlo Casino in Monaco, where games of chance use random numbers to determine the outcome.

The Monte Carlo method typically involves simulating a large number of random samples or scenarios to generate estimates of complex systems or problems that are difficult to solve analytically. These random samples are used to estimate probabilities or expected values of the system or problem under investigation.

For example, in physics, the Monte Carlo method is used to simulate the behavior of particles in a system by generating random positions and velocities for each particle and then computing the resulting behavior of the system. In finance, Monte Carlo simulations are used to estimate the value of financial instruments such as options or bonds, by simulating a large number of possible future scenarios and calculating the expected value of the instrument under each scenario.

The Monte Carlo method can be particularly useful in situations where the problem is too complex to be solved analytically, and there are many sources of randomness or uncertainty involved. However, the accuracy of Monte Carlo simulations depends on the number of samples or scenarios simulated, and in some cases, the method can be computationally expensive.

We can simulate the Hire Assistant problem using the Monte Carlo method, which is a probabilistic algorithm that uses random sampling to obtain numerical results.

Here's how we can use Monte Carlo method to simulate the Hire Assistant problem:

1. Generate a large number of candidate pools, each containing a random permutation of the same set of candidates.
2. For each candidate pool, run the Hire Assistant algorithm on the candidates and record the total cost of hiring and firing assistants.
3. Compute the average cost over all the candidate pools to obtain an estimate of the expected cost.


## Law of Large Numbers

* https://en.wikipedia.org/wiki/Law_of_large_numbers

![Fair Dice](https://upload.wikimedia.org/wikipedia/commons/thumb/c/c9/Lawoflargenumbers.svg/450px-Lawoflargenumbers.svg.png)
### Wisdom of the crowds

### Reversal to the mean


In [ ]:
throws = 1
random.seed(2025)
throw_dict = {}
for _ in range(7):
    throws *= 10
    # print(f"Throwing  dice{throws} times")
    dice_throws = [random.randint(1,6) for _ in range(throws)]
    # throw_dict[throws] = dice_throws
    avg = sum(dice_throws) / throws
    throw_dict[str(throws)] = avg
    print(f"Average dice from {throws} is {avg}")


In [ ]:
# let's plot the dictionary
import matplotlib.pyplot as plt
plt.plot(throw_dict.keys(), throw_dict.values())
plt.show()


In [ ]:
random.sample([1,2,3,4], 4) # we return a random sample without replacement - meaning we do not get doubles


In [ ]:
## Monte Carlo Simulation

def hire_assistant_simulate(
    candidates,
    hire_cost,
    fire_cost,
    interview_cost,
    num_simulations,
    is_hirable_fun=is_hirable,
    max_budget=0,
    debug=False,
    seed=SEED,
):
    total_cost = 0
    total_score = 0
    n = len(candidates)
    local_rng = random.Random(seed)

    for _ in range(num_simulations):
        candidate_pool = local_rng.sample(candidates, n)
        cost, best_candidate, best_score = hire_assistant(
            candidate_pool,
            is_hirable_fun,
            hire_cost,
            fire_cost,
            interview_cost,
            max_budget=max_budget,
            debug=debug,
        )
        total_cost += cost
        total_score += best_score

    return total_cost / num_simulations, total_score / num_simulations


In [ ]:
# so we will calculate the average cost of hiring assistant for 10_000 simulations
# this is using the basic is_hirable function the default one
hire_assistant_simulate(candidates, hire_cost=2_000, fire_cost=5_000, interview_cost=200, num_simulations=10_000)
# not surprisingly we always get the best candidate, because we are checking every applicant


In [ ]:
# let's try the threshold function
hire_assistant_simulate(candidates, hire_cost=2_000, fire_cost=5_000, interview_cost=200, num_simulations=10_000, is_hirable_fun=is_hirable_thresh)
# so knowing a good threshold is important, because if you set it too high you might not hire anyone
# if you set it too low you might hire too many people


In [ ]:
# let's make a new is_hirable_thresh function with 0.70 threshold
# let's make a new is_hirable function that will simply check for improvement and also for threshold score say 0.9
def is_hirable_thresh_07(candidate, index, num_candidates, best_score, hire_cost, fire_cost, total_cost, max_budget, threshold=0.7):
    return candidate > best_score and candidate > threshold

hire_assistant_simulate(candidates, hire_cost=2_000, fire_cost=5_000, interview_cost=200, num_simulations=10_000, is_hirable_fun=is_hirable_thresh_07)


In [ ]:
threshold_rows = []
for threshold in (0.50, 0.70, 0.80, 0.90, 0.95):
    threshold_rule = make_hirable_function_with_threshold(threshold)
    average_cost, average_score = hire_assistant_simulate(
        candidates,
        hire_cost=2_000,
        fire_cost=5_000,
        interview_cost=200,
        num_simulations=10_000,
        is_hirable_fun=threshold_rule,
    )
    threshold_rows.append({
        "threshold": threshold,
        "average_cost": average_cost,
        "average_score": average_score,
    })

show_table(threshold_rows)


## When the Score Distribution Is Not Known: The Secretary Problem

The classical secretary problem is stricter than the hiring-assistant problem:

- candidates arrive one at a time in random order;
- after each interview we know only the candidate's rank relative to the candidates already seen;
- we may hire exactly one candidate;
- rejected candidates cannot be recalled;
- the goal is to maximize the probability of hiring the single best candidate.

The standard **look-then-leap** strategy is:

1. Observe the first `r` candidates, but hire none of them.
2. Remember the best candidate from this observation phase.
3. Hire the first later candidate who is better than everyone observed so far.
4. If no such candidate appears, hire the last candidate.

For large `n`, choosing `r` close to `n/e` maximizes the success probability. If `t = r/n`, the approximate success probability is `-t ln(t)`, which is maximized at `t = 1/e` and has value about `1/e = 0.3679`.

Sources to compare with this simulation:

- Changyao Chen, [The Secretary Problem](https://changyaochen.github.io/secretary-problem/)
- Thomas S. Ferguson, [Who Solved the Secretary Problem?](https://changyaochen.github.io/assets/pdfs/who_solved_secretary_problem.pdf)
- P. R. Freeman, [The Secretary Problem and its Extensions: A Review](https://changyaochen.github.io/assets/pdfs/secprob2.pdf)


In [ ]:
def secretary_select(order, skip):
    """Return the selected index and value using the look-then-leap rule."""
    n = len(order)
    if n == 0:
        raise ValueError("order must contain at least one candidate")

    skip = max(0, min(skip, n - 1))
    best_observed = max(order[:skip], default=-math.inf)

    for index in range(skip, n):
        if order[index] > best_observed:
            return index, order[index]

    return n - 1, order[-1]


def secretary_theoretical_success(n, skip):
    """Exact success probability for the look-then-leap rule with n candidates."""
    if n <= 0:
        raise ValueError("n must be positive")

    skip = max(0, min(skip, n - 1))
    if skip == 0:
        return 1 / n

    return (skip / n) * sum(1 / (position - 1) for position in range(skip + 1, n + 1))


def secretary_asymptotic_success(skip_fraction):
    """Large-n approximation -t ln(t), where t is the skipped fraction."""
    if skip_fraction <= 0 or skip_fraction >= 1:
        return 0.0
    return -skip_fraction * math.log(skip_fraction)


def simulate_secretary_strategy(n, skip, trials=10_000, seed=SEED):
    local_rng = random.Random(seed)
    base_order = list(range(n))
    successes = 0
    selected_positions = []
    selected_ranks = []

    for _ in range(trials):
        order = base_order[:]
        local_rng.shuffle(order)
        selected_index, selected_value = secretary_select(order, skip)
        successes += selected_value == n - 1
        selected_positions.append(selected_index + 1)
        selected_ranks.append(n - selected_value)

    return {
        "n": n,
        "skip": skip,
        "skip_fraction": skip / n,
        "trials": trials,
        "empirical_success": successes / trials,
        "theoretical_success": secretary_theoretical_success(n, skip),
        "asymptotic_success": secretary_asymptotic_success(skip / n),
        "average_selected_position": sum(selected_positions) / trials,
        "average_selected_rank": sum(selected_ranks) / trials,
    }


n = 100
skip = round(n / math.e)
simulate_secretary_strategy(n, skip, trials=20_000)


In [ ]:
def secretary_success_curve(n=100, trials=3_000, step=2, seed=SEED):
    rows = []
    for skip in range(0, n, step):
        rows.append(simulate_secretary_strategy(n, skip, trials=trials, seed=seed + skip))
    return rows


curve_rows = secretary_success_curve(n=100, trials=3_000, step=2)
best_empirical = max(curve_rows, key=lambda row: row["empirical_success"])
best_theoretical = max(curve_rows, key=lambda row: row["theoretical_success"])

print(f"Best empirical skip in sampled grid: {best_empirical['skip']} / 100")
print(f"Best theoretical skip in sampled grid: {best_theoretical['skip']} / 100")
print(f"n/e suggests skipping about {round(100 / math.e)} candidates")

show_table([
    row for row in curve_rows
    if row["skip"] in {0, 10, 20, 30, 36, 38, 50, 70, 90}
])


In [ ]:
if plt is not None:
    skip_values = [row["skip"] for row in curve_rows]
    empirical = [row["empirical_success"] for row in curve_rows]
    theoretical = [row["theoretical_success"] for row in curve_rows]

    plt.figure(figsize=(8, 4.5))
    plt.plot(skip_values, empirical, "o", label="simulation")
    plt.plot(skip_values, theoretical, "-", label="exact theory")
    plt.axvline(100 / math.e, color="tab:red", linestyle="--", label="n/e")
    plt.xlabel("candidates skipped before hiring is allowed")
    plt.ylabel("probability of selecting the best candidate")
    plt.title("Secretary problem: look-then-leap strategy")
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()
else:
    print("matplotlib is not available, skipping plot")


In [ ]:
def show_secretary_experiment(n=100, skip_fraction=1 / math.e, trials=2_000):
    skip = round(n * skip_fraction)
    result = simulate_secretary_strategy(n=n, skip=skip, trials=trials)
    print(f"n = {n}, skip = {skip}, trials = {trials}")
    print(f"empirical success:   {result['empirical_success']:.4f}")
    print(f"exact theory:        {result['theoretical_success']:.4f}")
    print(f"large-n estimate:    {result['asymptotic_success']:.4f}")
    print(f"average rank chosen: {result['average_selected_rank']:.2f} (1 means best)")


if WIDGETS_AVAILABLE:
    widgets.interact(
        show_secretary_experiment,
        n=widgets.IntSlider(value=100, min=10, max=300, step=10),
        skip_fraction=widgets.FloatSlider(value=1 / math.e, min=0.0, max=0.95, step=0.01, readout_format=".2f"),
        trials=widgets.IntSlider(value=2_000, min=500, max=10_000, step=500),
    )
else:
    show_secretary_experiment()


## Other Versions of the Secretary Problem

The classical version is only one model. Common variants include:

- **Multiple hires**: choose up to `k` candidates instead of exactly one.
- **Hiring and firing**: allow replacement, but charge a cost for each replacement.
- **Unknown number of applicants**: make decisions without knowing `n` in advance.
- **Adversarial order**: remove the random-arrival assumption and study what guarantees remain.
- **Distributed selection**: coordinate several decision-makers or offices.

These variants connect the secretary problem to online algorithms, optimal stopping, and realistic hiring models such as the Stanford hiring-problem slides summarized in `sources/randomized_algorithms/hiring_problem.md`.


## Side Story - Calculating Pi via Monte Carlo Method

To calculate the value of pi using Monte Carlo sampling, generate random points in a square and count how many fall inside the inscribed circle. The ratio of hits inside the circle estimates the area ratio, so `pi` is approximately four times that ratio.

This is not a good practical way to compute pi, but it is a useful demonstration of convergence by repeated random sampling.


![Circle](https://upload.wikimedia.org/wikipedia/commons/thumb/8/84/Pi_30K.gif/440px-Pi_30K.gif)


In [ ]:
# import random

def estimate_pi(num_points):
    num_points_in_circle = 0
    for _ in range(num_points):
        x = random.uniform(-1, 1)
        y = random.uniform(-1, 1)
        if x**2 + y**2 <= 1: # this is kind of slow because of the square root
            num_points_in_circle += 1
    pi_estimate = 4 * num_points_in_circle / num_points
    return pi_estimate

# This code takes in the number of points to generate,
#  generates random points within a square of length 2 centered at the origin,
# counts the number of points that lie inside the quarter-circle of radius 1 centered at the origin,
# estimates the area of the quarter-circle as the proportion of points
# inside the quarter-circle to the total number of points generated,
# and estimates the value of pi as four times the estimated area of the quarter-circle.
#  The more points generated, the more accurate theb estimate of pi will be.


In [ ]:
throws = 1
for _ in range(7):
    throws *= 10
    # print(f"Throwing  dice{throws} times")
    print(f"Average PI from {throws} pins is {estimate_pi(throws)}")

# so takes about 10 Million throws to get 2 digits of precision, not very practical for PI but still useful in general


## Side story: The Monty Hall Problem

![Goat](https://upload.wikimedia.org/wikipedia/commons/thumb/3/3f/Monty_open_door.svg/440px-Monty_open_door.svg.png)


The Monty Hall problem is a famous probability puzzle that is named after the host of the game show "Let's Make a Deal," Monty Hall. The problem is based on a hypothetical game show where a contestant is presented with three doors. Behind one of the doors is a valuable prize, while the other two doors hide goats.

The contestant chooses one of the three doors, but before the chosen door is opened, the host (Monty Hall) opens one of the other two doors to reveal a goat. The contestant is then given the option to stick with their original choice or switch to the other unopened door.

The question is whether the contestant should stick with their original choice or switch to the other door in order to increase their chances of winning the prize. The answer may seem counterintuitive, but switching actually increases the contestant's chances of winning the prize from 1/3 to 2/3. This is because when the contestant first made their choice, they had a 1/3 chance of being correct. When the host opened one of the other doors to reveal a goat, the remaining unopened door had a 2/3 chance of hiding the prize.


### Correct Strategy for Monty Hall Problem

The correct strategy is to switch. Your first choice wins with probability `1/3`. With probability `2/3`, your first choice is a goat; then the host is forced to reveal the other goat, and switching wins the car.

So staying wins with probability `1/3`, while switching wins with probability `2/3`.

Famously, the question was discussed in *Parade* magazine in 1990 when Marilyn vos Savant explained the switching strategy.

Wiki: https://en.wikipedia.org/wiki/Monty_Hall_problem


In [ ]:
def monty_hall_simulation(switch):
    doors = ["goat", "goat", "car"]
    random.shuffle(doors)
    chosen_door = random.choice(doors)
    if chosen_door == "car":
        if switch: # so we chose the switch strategy and were unlucky to have chosen the car already - so we get goat
            return 0
        else: # no switch strategy - stay put
            return 1
    else: # when we have chosen a goat
        if switch: # we apply switch strategy
            return 1  # we win the car
        else:  # stay put strategy fails here - we end up with the goat
            return 0

num_simulations = 1_000_000
switch = True
wins = 0

for i in range(num_simulations):
    wins += monty_hall_simulation(switch)

print(f"Probability of winning with switch: {wins / num_simulations:.4f}")
print(f"Probability of winning without switch: {(num_simulations - wins) / num_simulations:.4f}")


In [ ]:
num_simulations = 1_000_000
switch = False
wins = 0

for i in range(num_simulations):
    wins += monty_hall_simulation(switch)

print(f"Probability of winning without switch: {wins / num_simulations:.4f}")
print(f"Probability of winning WITH switch: {(num_simulations - wins) / num_simulations:.4f}")


## Using random simulation to obtain answers

So if you have trouble coming up with an answer to some algorith, you can use simulation to come up with a good aproximation.

Key idea is to have some knowledge of distribution of inputs.


## Jupyter %%timeit also works on this principle


In [ ]:
%%timeit
sorted(list(range(1_000_000)))


## Optimal Stopping Problem

The secretary problem is an example of an **optimal stopping problem**: after each observation, the algorithm must either stop and accept the current candidate or continue and lose that candidate forever.

For the classical one-choice secretary problem with random arrival order, the best threshold rule observes about `n/e` candidates and then accepts the first later candidate who beats all observed candidates. This does not guarantee success on a single run. It maximizes the probability of selecting the best candidate, and that maximum probability approaches `1/e` as `n` grows.


## Other Algorithms that use randomness

### Matrix Operations

* https://tropp.caltech.edu/notes/Tro20-Randomized-Algorithms-LN.pdf
* There are some Latvian researchers that have done work in this field
* https://en.wikipedia.org/wiki/Freivalds%27_algorithm
* https://en.wikipedia.org/wiki/R%C5%ABsi%C5%86%C5%A1_M%C4%81rti%C5%86%C5%A1_Freivalds


In [ ]:
## TODO more randomized algorithms
